# 02 · Preparación de los datos

**Proyecto:** SCA-2026-01 · Herramienta de segmentación de clientes de Costanera Andina
**Etapa CRISP-DM y paquetes EDT:** Preparación de los datos (transversal) · 2.3.1 Pipeline de preparación y ambiente analítico · 2.3.3 Actualización en cada iteración
**Iteración:** Iteración 1 (26/10 a 27/11) y ajustes de la iteración 2
**Equipo:** Felipe Ahumada Silva (Jefe de Proyecto) · Francisca Carrasco Lozano (Analista de Datos líder)

**Objetivo de este notebook:** dejar una base analítica seudonimizada, versionada y reproducible, que es la que usan los modelos.

**Requerimientos que se trabajan:** RNF.3 (datos confiables), RNF.4 (privacidad: sin identificadores), RNF.8 (seguridad), RF.10 (recálculo).

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from costanera import carga, config, evaluacion, modelado, preparacion
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
FIG = config.FIGURAS
FIG.mkdir(parents=True, exist_ok=True)

## 1. Seudonimización y variables derivadas
El identificador original se reemplaza por un seudónimo (SHA-256 con sal). La sal se guarda fuera del repositorio en producción. Se agrega el tramo etario para la revisión de equidad.

In [2]:
datos, resumen = carga.cargar()
base = preparacion.preparar(datos)
print("¿Queda el identificador original?", "id_cliente" in base.columns)
print("¿Seudónimos únicos?", base["seudonimo"].is_unique)
base.head()

¿Queda el identificador original? False
¿Seudónimos únicos? True


,seudonimo,genero,edad,educacion,estado_civil,ingreso_anual_k,indice_gasto,tramo_etario
0,956c777e7a09,M,19,High School,Married,15,39,18-29
1,4165771da49a,M,21,Graduate,Single,15,81,18-29
2,7e223ce934bc,F,20,Graduate,Married,16,6,18-29
3,0a84d931fa04,F,23,High School,Unknown,16,77,18-29
4,26bd93118514,F,31,Uneducated,Married,17,40,30-44


## 2. Corte con consentimiento vigente (2.3.2, en curso)
La Ley 21.719 rige desde el 1/12/2026. El pipeline ya tiene el filtro `solo_con_consentimiento`, pero la extracción actual no trae la columna de consentimiento: llegará con los resultados de la campaña de renovación (22/12). Por eso la iteración 2 trabaja con la base completa y la versión final (iteración 3) se construirá solo con titulares con consentimiento.

In [3]:
try:
    preparacion.preparar(datos, solo_con_consentimiento=True)
except ValueError as e:
    print("Control activo ->", e)

Control activo -> El archivo no trae la columna 'consentimiento'; no se puede hacer el corte 2.3.2.


## 3. Escalamiento
Ingreso e índice de gasto están en escalas distintas; K-means usa distancias, así que se estandarizan (media 0, desviación 1).

In [4]:
X, escalador = preparacion.escalar(base)
pd.DataFrame(X, columns=config.VARIABLES_MODELO).describe().round(2)

,ingreso_anual_k,indice_gasto
count,200.00,200.00
mean,-0.00,-0.00
std,1.00,1.00
min,-1.74,-1.91
25%,-0.73,-0.60
50%,0.04,-0.01
75%,0.67,0.89
max,2.92,1.89


## 4. Base analítica versionada
Cada iteración guarda su versión. Se verifica que se pueda reconstruir igual (RF.10).

In [5]:
for v in ["v1"]:
    print(Path(preparacion.guardar_base(base, v)).relative_to(config.RAIZ))
otra = preparacion.preparar(carga.cargar()[0])
print("¿El pipeline reconstruye la misma base?", otra.equals(base))

data/processed/base_analitica_v1.csv
¿El pipeline reconstruye la misma base? True
